# Kafka and PySpark Streaming Example

This notebook demonstrates how to read messages from a Kafka topic, display the messages, and parse JSON data using PySpark. 

## Initialization

First, we initialize a Spark session with the necessary Kafka package.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StringType
import time

spark = SparkSession.builder \
    .appName("KafkaSparkStreamingExample") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0")\
    .getOrCreate()

26/05/18 18:32:19 WARN Utils: Your hostname, codespaces-711bff resolves to a loopback address: 127.0.0.1; using 10.0.13.154 instead (on interface eth0)
26/05/18 18:32:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/sdkman/candidates/spark/3.5.1/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vscode/.ivy2/cache
The jars for the packages stored in: /home/vscode/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-591e14da-4311-4f68-bdce-9628dae14fc0;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
downloading https://repo1.maven.org/maven2/org/apache/spark/spa

## Event Data, JSON Data Structure and Schema

We are dealing with a store submitting order updates. 
This store provides details such as order time, order ID, item ID, order units, and address information (city, state, and zipcode).


The Event, JSON data we are dealing with has the following structure:

```json
{
  "ordertime": 1516952774174,
  "orderid": 41,
  "itemid": "Item_7",
  "orderunits": 7.1615426137476,
  "address": {
    "city": "City_7",
    "state": "State_",
    "zipcode": 39126
  }
}


This JSON represents an order with the following fields:

* ordertime: The timestamp of the order.
* orderid: The ID of the order.
* itemid: The ID of the item.
* orderunits: The number of units ordered.
* address: A **nested JSON object** containing the address details (city, state, zipcode).


#### Streaming and loading the data into a dataframe

We have set up a streaming DataFrame `df` to continuously read data from the specified Kafka topic.
Here’s how we do it:

You're configuring the Kafka parameters and creating a Spark streaming DataFrame that subscribes to a Kafka topic using PySpark. Here’s a breakdown of the steps and components involved:

For now, we assume that the topic is a continuous stream of JSON-formatted messages representing commerce transactions, where each message contains details about individual orders.


### Kafka Parameters Setup

- **`kafka.bootstrap.servers`**: This specifies the Kafka server's address, including the port. It's used by Spark to locate the Kafka cluster.
- **`subscribe`**: This parameter indicates the name of the Kafka topic (`commerce-fhtw`) that Spark should subscribe to. It is crucial for directing Spark to the right stream of data.
- **`kafka.security.protocol`**: Set to `SASL_SSL` to indicate that the connection to Kafka should be secured using SASL (Simple Authentication and Security Layer) over SSL (Secure Sockets Layer).
- **`kafka.sasl.mechanism`**: This specifies the SASL mechanism to use for authentication; in this case, `PLAIN`, which means it will use a simple username and password method.
- **`kafka.sasl.jaas.config`**: Provides the

kcat is a free tool

In [7]:
#IP address is the kcat server
# a stream of orders is generated in real-time
# goal: how can we process, analyse, query, trigger events using sparks
kafka_params = {
      "kafka.bootstrap.servers": "46.225.20.89:9092",
      "subscribe": "commerce-fhtw",
      "kafka.security.protocol": "PLAINTEXT",
      "startingOffsets": "latest",
}

similar real-time music events
or stocks 

project at the end of the course: write an algorithm that is capable of recommanding songs in real-time
how to process data that is constantly comming in?

In [8]:
df = spark \
    .readStream \
    .format("kafka") \
    .options(**kafka_params) \
    .load()

# How does the data look like?

We initiate the process by displaying all incoming messages to gain a preliminary understanding of the data flow. This step provides a basic overview of the ongoing activity within the Kafka topic.

In [ ]:
query = df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

26/05/18 18:44:03 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-b172eefc-f9d5-49ea-afd7-a201f1b652e3. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/18 18:44:03 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/05/18 18:44:05 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+---+-----+-----+---------+------+---------+-------------+
|key|value|topic|partition|offset|timestamp|timestampType|
+---+-----+-----+---------+------+---------+-------------+
+---+-----+-----+---------+------+---------+-------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+--------------------+-------------+---------+------+--------------------+-------------+
|                key|               value|        topic|partition|offset|           timestamp|timestampType|
+-------------------+--------------------+-------------+---------+------+--------------------+-------------+
|[36 30 30 30 32 35]|[7B 22 6F 72 64 6...|commerce-fhtw|        0| 10498|2026-05-18 18:44:...|            0|
|[36 30 30 30 32 36]|[7B 22 6F 72 64 6...|commerce-fhtw|        0| 10499|2026-05-18 18:44:...|            0|
|[36 30 30 30 32 37]|[7B 22 6F 72 64 6...|commerce-fhtw|        0| 10500|2026-05-18 18:44:...|            0|
+-------------------+--------------------+-------------+---------+------+--------------------+-------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+-------------------+--------------------+-------------+---------+------+--------------------+-------------+
|                key|               value|        topic|partition|offset|           timestamp|timestampType|
+-------------------+--------------------+-------------+---------+------+--------------------+-------------+
|[36 30 30 30 32 38]|[7B 22 6F 72 64 6...|commerce-fhtw|        0| 10501|2026-05-18 18:44:...|            0|
|[36 30 30 30 32 39]|[7B 22 6F 72 64 6...|commerce-fhtw|        0| 10502|2026-05-18 18:44:...|            0|
|[36 30 30 30 33 30]|[7B 22 6F 72 64 6...|commerce-fhtw|        0| 10503|2026-05-18 18:44:...|            0|
+-------------------+--------------------+-------------+---------+------+--------------------+-------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+-------------------+--------------------+-------------+---------+------+--------------------+-------------+
|                key|               value|        topic|partition|offset|           timestamp|timestampType|
+-------------------+--------------------+-------------+---------+------+--------------------+-------------+
|[36 30 30 30 33 31]|[7B 22 6F 72 64 6...|commerce-fhtw|        0| 10504|2026-05-18 18:44:...|            0|
|[36 30 30 30 33 32]|[7B 22 6F 72 64 6...|commerce-fhtw|        0| 10505|2026-05-18 18:44:...|            0|
+-------------------+--------------------+-------------+---------+------+--------------------+-------------+

-------------------------------------------
Batch: 4
-------------------------------------------
+-------------------+--------------------+-------------+---------+------+--------------------+-------------+
|                key|               value|


This output contains several important pieces of information for each message:

- **`key`**: The key of the message in the Kafka topic, represented in a byte array format.
- **`value`**: The actual content of the message, also in a byte array format. This is where our JSON data is stored.
- **`topic`**: The name of the Kafka topic from which the message was read.
- **`partition`**: The partition number in the Kafka topic where the message was stored.
- **`offset`**: The offset of the message within the partition, which serves as a unique identifier for the message within that partition.
- **`timestamp`**: The timestamp indicating when the message was produced.
- **`timestampType`**: The type of timestamp (e.g., whether it is the creation time or log append time).

By examining this output, you can get a basic idea of the structure of the messages coming from the Kafka topic, including the metadata and the actual data content.


As you will see the messages are keep coming in and coming in  and we want to terminate that in order to allow for another DF to process.

In [ ]:
query.stop()

26/05/18 18:44:21 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 11, writer: ConsoleWriter[numRows=20, truncate=true]] is aborting.
26/05/18 18:44:21 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 11, writer: ConsoleWriter[numRows=20, truncate=true]] aborted.


26/05/18 18:44:21 ERROR Utils: Aborting task
org.apache.spark.TaskKilledException
	at org.apache.spark.TaskContextImpl.killTaskIfInterrupted(TaskContextImpl.scala:267)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:36)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:43)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask.$anonfun$run$1(WriteToDataSourceV2Exec.scala:441)
	at org.apache.spark.util.Utils$.tryWithSafeFinallyAndFailureCallbacks(Utils.scala:1397)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask.run(WriteToDataSourceV2Exec.s

Now, in order to make our lives easier, we will write a function that allows us to display all the incoming data instead of having to write this whole code section every now and then.


In [11]:
def display_stream(df, timeout=10):
    query = df.writeStream \
        .outputMode("append") \
        .format("console") \
        .option("truncate", "false") \
        .start()
    # Let the stream run for a short period and then stop (for demonstration purposes)
    time.sleep(timeout)
    query.stop()

### Read the json output and event data

Now, we actually want to analyze the section that contains the value, which holds the actual Kafka message and event data. As you can see, the message content is in the `value` entry. Therefore, we need to read and process the data from the `value` entry.

Here’s how we can do this:

In [12]:
# Select and cast the key and value columns to STRING
df_message = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")

display_stream(df_message, 10)

26/05/18 18:44:34 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-ce0a3f2c-e82f-412c-b207-93021010b96d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/18 18:44:34 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/18 18:44:35 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+---+-----+
|key|value|
+---+-----+
+---+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key   |value                                                                                                                                                                  |
+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|600055|{"ordertime": 1779129876032, "orderid": 600055, "itemid": "coffee-beans", "orderunits": 2.91, "address": {"city": "Linz", "state": "Oberoesterreich", "zipcode": 4020}}|
+------+-----------------------------------------

-------------------------------------------
Batch: 3
-------------------------------------------
+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+
|key   |value                                                                                                                                                      |
+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+
|600057|{"ordertime": 1779129878033, "orderid": 600057, "itemid": "green-tea", "orderunits": 1.84, "address": {"city": "Vienna", "state": "Wien", "zipcode": 1010}}|
+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+

-------------------------------------------
Batch: 4
------------------------

-------------------------------------------
Batch: 6
-------------------------------------------
+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+
|key   |value                                                                                                                                                      |
+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+
|600060|{"ordertime": 1779129881034, "orderid": 600060, "itemid": "green-tea", "orderunits": 3.62, "address": {"city": "Vienna", "state": "Wien", "zipcode": 1010}}|
+------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+



-------------------------------------------
Batch: 7
-------------------------------------------
+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key   |value                                                                                                                                                         |
+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------+
|600061|{"ordertime": 1779129882035, "orderid": 600061, "itemid": "water-bottle", "orderunits": 3.92, "address": {"city": "Vienna", "state": "Wien", "zipcode": 1010}}|
+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------+



-------------------------------------------
Batch: 8
-------------------------------------------
+------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key   |value                                                                                                                                                          |
+------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|600062|{"ordertime": 1779129883035, "orderid": 600062, "itemid": "desk-lamp", "orderunits": 2.38, "address": {"city": "Innsbruck", "state": "Tirol", "zipcode": 6020}}|
+------+---------------------------------------------------------------------------------------------------------------------------------------------------------------+

-------------------------------------------
Batch: 9
----

## Extract the actual data from the json

Alright, now we get the actual JSON message, but we want to deal with it like a data frame. So we would actually like to extract the data inside the JSON data as a column.
as a DF.


### The Schema
We define the schema for the incoming JSON data. This schema matches the structure of the JSON messages we expect from Kafka.

The schema is defined to match the structure of the incoming JSON data. It specifies the data types for each field in the JSON object.
This is necessary because it allows Spark to correctly parse and interpret the data, ensuring that each field is handled appropriately (e.g., as an integer, string, etc.).
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType

In [13]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, IntegerType
# Define the schema for the JSON data
schema = StructType([
    StructField("ordertime", LongType()),
    StructField("orderid", IntegerType()),
    StructField("itemid", StringType()),
    StructField("orderunits", DoubleType()),
    StructField("address", StructType([
        StructField("city", StringType()),
        StructField("state", StringType()),
        StructField("zipcode", IntegerType())
    ]))
])

In [ ]:
df_value = df.selectExpr("CAST(value AS STRING) as json_string")
# Parse the JSON data and apply schema
parsed_df = df_value.withColumn("jsonData", from_json(col("json_string"), schema)).select("jsonData.*")
display_stream(parsed_df, 10)

26/05/18 18:44:58 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-d1c4b5f2-e9af-40d4-8b20-aefcb933155c. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/18 18:44:58 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/18 18:44:59 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-------+------+----------+-------+
|ordertime|orderid|itemid|orderunits|address|
+---------+-------+------+----------+-------+
+---------+-------+------+----------+-------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------------+-------+---------+----------+--------------------+
|ordertime    |orderid|itemid   |orderunits|address             |
+-------------+-------+---------+----------+--------------------+
|1779129900041|600079 |desk-lamp|1.24      |{Vienna, Wien, 1010}|
+-------------+-------+---------+----------+--------------------+

-------------------------------------------
Batch: 2
-------------------------------------------
+-------------+-------+---------+----------+------------------------+
|ordertime    |orderid|itemid   |orderunits|address                 |
+-------------+-------+---------+----------+------------------------+
|1779129901042|600080 |desk-lamp|1.32      |{Graz, Steiermark, 8010}|
+-------------+-------+---------+----------+------------------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+-------------+-------+---------+----------+--------------------+
|ordertime    |orderid|itemid   |orderunits|address             |
+-------------+-------+---------+----------+--------------------+
|1779129902042|600081 |usb-cable|6.78      |{Vienna, Wien, 1010}|
+-------------+-------+---------+----------+--------------------+

-------------------------------------------
Batch: 4
-------------------------------------------
+-------------+-------+---------+----------+------------------------+
|ordertime    |orderid|itemid   |orderunits|address                 |
+-------------+-------+---------+----------+------------------------+
|1779129903042|600082 |usb-cable|9.89      |{Innsbruck, Tirol, 6020}|
+-------------+-------+---------+----------+------------------------+

-------------------------------------------
Batch: 5
-------------------------------------------
+-------------+-------+----

-------------------------------------------
Batch: 7
-------------------------------------------
+-------------+-------+---------+----------+------------------------+
|ordertime    |orderid|itemid   |orderunits|address                 |
+-------------+-------+---------+----------+------------------------+
|1779129906043|600085 |desk-lamp|1.57      |{Graz, Steiermark, 8010}|
+-------------+-------+---------+----------+------------------------+

-------------------------------------------
Batch: 8
-------------------------------------------
+-------------+-------+---------+----------+--------------------------------------+
|ordertime    |orderid|itemid   |orderunits|address                               |
+-------------+-------+---------+----------+--------------------------------------+
|1779129907044|600086 |green-tea|4.52      |{St. Poelten, Niederoesterreich, 3100}|
+-------------+-------+---------+----------+--------------------------------------+



26/05/18 18:45:09 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 9, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
26/05/18 18:45:09 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 9, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.


26/05/18 18:45:09 ERROR Utils: Aborting task
org.apache.spark.TaskKilledException
	at org.apache.spark.TaskContextImpl.killTaskIfInterrupted(TaskContextImpl.scala:267)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:36)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:460)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:43)
	at org.apache.spark.sql.execution.datasources.v2.WritingSparkTask.$anonfun$run$1(WriteToDataSourceV2Exec.scala:441)
	at org.apache.spark.util.Utils$.tryWithSafeFinallyAndFailureCallbacks(Utils.scala:1397)
	at org.apache.spark.sql.exec

## Analyzing the data

So since just streaming and displaying the data is quite boring, we want to actually do some stuff with it. Let's try to deal with it.


In [ ]:
from pyspark.sql.functions import sum

# Aggregate the order units
order_units_df = parsed_df.groupBy().agg(sum("orderunits").alias("total_order_units"))

# Display the aggregated data
# complete mode is required for aggregations!!
query = order_units_df.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

time.sleep(20)
query.stop()

26/05/18 18:45:21 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-bb09be87-7a5f-41e1-aa22-a59e4a1031cc. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/18 18:45:21 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/05/18 18:45:21 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------------+
|total_order_units|
+-----------------+
|             NULL|
+-----------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-----------------+
|total_order_units|
+-----------------+
|6.140000000000001|
+-----------------+

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------+
| total_order_units|
+------------------+
|10.370000000000001|
+------------------+

-------------------------------------------
Batch: 3
-------------------------------------------
+------------------+
| total_order_units|
+------------------+
|11.490000000000002|
+------------------+

-------------------------------------------
Batch: 4
-------------------------------------------
+------------------+
| total_order_units|
+------------------+
|18.270000000000003|
+------------------+

-------------------------------------------
Batch: 5
-------------------------------------------
+------------------+
| total_order_units|
+------------------+
|24.020000000000003|
+-----------

-------------------------------------------
Batch: 12
-------------------------------------------
+-----------------+
|total_order_units|
+-----------------+
|            67.53|
+-----------------+

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------------+
|total_order_units|
+-----------------+
|            98.68|
+-----------------+

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------------+
|total_order_units|
+-----------------+
|           121.04|
+-----------------+



26/05/18 18:45:41 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 15, writer: ConsoleWriter[numRows=20, truncate=true]] is aborting.
26/05/18 18:45:41 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 15, writer: ConsoleWriter[numRows=20, truncate=true]] aborted.


26/05/18 18:45:41 WARN TaskSetManager: Lost task 0.0 in stage 63.0 (TID 62) (100eed67-482a-490d-8ae1-1b9246b3d37f.internal.cloudapp.net executor driver): TaskKilled (Stage cancelled: Job 47 cancelled part of cancelled job group f0d77544-5ac0-4b45-a704-a789be36642f)


### Your Turn

Try to calculate the orders per city!

In [ ]:
# Your task is to calculate the total order units per city

Calculating Average Order Units